# 08 — The model and its parameters, in PyTorch

Notebook 7 showed the framework runs the same net you built by hand. This notebook is the finale's
**estimator notebook** — the counterpart of "the method and its parameters" from every earlier chapter.
We take the real PyTorch network and **turn every knob**, each one introduced from the concept that
owns it, reading honestly what it does and how it fails. By the end you can *drive* a neural network,
not only run one.

You **read and turn** the knobs; the training harness is shown, once, and reused throughout.

**Prerequisites**
- Notebooks 1–7 — depth (NB 2), vanishing/exploding (NB 3), init (NB 4), dropout (NB 5), batch norm
  (NB 6), and the PyTorch canonical loop + `.train()`/`.eval()` (NB 7).
- Chapter 11 NB 4 — the sklearn estimator notebook (the shape of this one); chapter 00 — over/underfitting.

**What you'll be able to do**
- Set **capacity, activation, initialization, optimizer, learning rate, dropout and batch norm** on a torch net.
- Read the **three failure modes** — underfitting, overfitting, optimization failure — off the loss curve.
- Tune honestly on a validation set and report **one** number on a sealed test.

## The knobs, and where each comes from

Every knob we turn here has a home earlier in the course — this notebook is where they become
*settings on one object*:

- **capacity** (depth × width) — this notebook (the honest version of NB 2's "depth helps");
- **`activation`** — chapter 12 c02 / NB 3 (sigmoid saturates on a deep net → vanishing gradient);
- **initialization** (`torch.nn.init`) — c03 / NB 4 (He for ReLU, Xavier for tanh);
- **`nn.Dropout`** — c05 / NB 5;  **`nn.BatchNorm`** — c04 / NB 6;
- the **optimizer** menu (SGD / momentum / Adam) — c07 (mostly new here);
- **learning rate**, epochs, batch size — chapter 11.

One reusable harness, many knobs. We build it once and turn it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ml_course import colors, viz

viz.use_course_style()

SEED = 0
# The reproducibility contract (NB 7), verified on this machine:
# seeds + deterministic algorithms + a single thread -> bit-identical runs on CPU.
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)


def reset_seed(seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)


ACTIVATIONS = {"relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}


def build_net(width, depth, activation="relu", dropout=0.0, batchnorm=False,
              init="default", in_dim=2):
    # each hidden block: Linear, optional BatchNorm, activation, optional Dropout.
    layers, d = [], in_dim
    for _ in range(depth):
        layers.append(nn.Linear(d, width, bias=not batchnorm))   # BN carries the bias (NB 6)
        if batchnorm:
            layers.append(nn.BatchNorm1d(width))
        layers.append(ACTIVATIONS[activation]())
        if dropout > 0.0:
            layers.append(nn.Dropout(dropout))
        d = width
    layers.append(nn.Linear(d, 2))
    net = nn.Sequential(*layers)
    if init in ("small", "he"):
        for m in net:
            if isinstance(m, nn.Linear):
                if init == "he":
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")  # He (NB 4)
                else:
                    nn.init.normal_(m.weight, std=0.01)  # deliberately too small
                if m.bias is not None:   # a Linear feeding BatchNorm has no bias
                    nn.init.zeros_(m.bias)
    return net


def train(net, Xtr, ytr, Xva, yva, optimizer="adam", lr=0.05, epochs=300, weight_decay=0.0):
    params = net.parameters()
    if optimizer == "adam":
        opt = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    elif optimizer == "momentum":
        opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        opt = torch.optim.SGD(params, lr=lr, weight_decay=weight_decay)
    Xt, yt = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr)
    Xv, yv = torch.tensor(Xva, dtype=torch.float32), torch.tensor(yva)
    tr_loss, va_loss = [], []
    for _ in range(epochs):
        net.train()
        opt.zero_grad()
        loss = F.cross_entropy(net(Xt), yt)
        loss.backward()
        opt.step()
        net.eval()
        with torch.no_grad():
            tr_loss.append(loss.item())
            va_loss.append(F.cross_entropy(net(Xv), yv).item())
    net.eval()
    with torch.no_grad():
        tr_acc = (net(Xt).argmax(1) == yt).float().mean().item()
        va_acc = (net(Xv).argmax(1) == yv).float().mean().item()
    return tr_loss, va_loss, tr_acc, va_acc


# Two datasets: 2-D for the trainable-knob demos; high-dim where dropout/BN bite (NB 5/6).
# (Scaled on all features for brevity; a strict pipeline fits the scaler on train only — ch 00.)
Xm, ym = make_moons(n_samples=400, noise=0.2, random_state=SEED)
Xm = StandardScaler().fit_transform(Xm)
Xm_tr, Xm_va, ym_tr, ym_va = train_test_split(Xm, ym, test_size=0.3, stratify=ym, random_state=SEED)

Xh, yh = make_classification(n_samples=1200, n_features=50, n_informative=10, n_redundant=10,
                             n_classes=2, class_sep=1.0, random_state=SEED)
Xh = StandardScaler().fit_transform(Xh)
Xh_tr, Xh_rest, yh_tr, yh_rest = train_test_split(
    Xh, yh, train_size=200, stratify=yh, random_state=SEED)
Xh_va, Xh_te, yh_va, yh_te = train_test_split(
    Xh_rest, yh_rest, train_size=500, test_size=500, stratify=yh_rest, random_state=SEED)
print(f"moons: train {Xm_tr.shape}  val {Xm_va.shape}")
print(f"high-dim: train {Xh_tr.shape}  val {Xh_va.shape}  test(sealed) {Xh_te.shape}")

## Capacity — too little underfits, too much overfits

The number of layers and units is the network's **capacity**. Too little and it cannot fit the data
(underfitting); too much, on limited data, and it memorizes (overfitting). This is the honest, measured
version of notebook 2's "depth helps" — here the inputs are 2-D.

In [ ]:
caps = {}
for name, width, depth in [("tiny  w2·d1", 2, 1), ("small w8·d1", 8, 1),
                           ("wide  w64·d1", 64, 1), ("deep  w32·d3", 32, 3)]:
    reset_seed()
    net = build_net(width, depth, activation="relu")
    _, _, tr, va = train(net, Xm_tr, ym_tr, Xm_va, ym_va, optimizer="adam", lr=0.05, epochs=300)
    caps[name] = (tr, va)
    n_params = sum(p.numel() for p in net.parameters())
    print(f"  {name}: train {tr:.3f}  val {va:.3f}  ({n_params} params)")

In [ ]:
names = list(caps)
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - 0.2, [caps[n][0] for n in names], 0.4, label="train", color=colors.COLORS["train"])
ax.bar(x + 0.2, [caps[n][1] for n in names], 0.4, label="validation", color=colors.COLORS["test"])
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel("accuracy")
ax.set_title("Capacity: too little underfits, too much opens a train–val gap")
ax.legend()
fig.tight_layout()
plt.show()

**Read the figure.** The tiny net (`w2·d1`) cannot even fit the training set — train and validation both
sit low: **underfitting**. Adding capacity lifts both, and the wide net reaches the best validation
accuracy. The deep net drives **training** accuracy to 1.0 but its **validation** accuracy dips below the
wide net's — the train–val gap has opened: **overfitting**. Capacity did the lifting; past "enough", more
of it stops helping the held-out score. (Notebook 2's honest lesson, now a knob.)

## `activation` — a tunable depth-pathology

Notebook 3 showed that on a deep network a saturating activation (sigmoid) makes the gradient vanish, so
training stalls. In torch the activation is a single argument. On a genuinely deep net (eight layers),
we flip it between ReLU, tanh, and sigmoid — holding everything else fixed — and watch.

In [ ]:
act_hist = {}
for act in ("relu", "tanh", "sigmoid"):
    reset_seed()
    net = build_net(16, 8, activation=act, init="he")
    tr_loss, _, tr, va = train(net, Xm_tr, ym_tr, Xm_va, ym_va, optimizer="sgd", lr=0.1, epochs=300)
    act_hist[act] = tr_loss
    print(f"  {act:8}: loss {tr_loss[0]:.3f} -> {tr_loss[-1]:.3f}   train {tr:.3f}  val {va:.3f}  "
          f"({'STALLED at ln 2' if tr_loss[-1] > 0.6 else 'trained'})")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
palette = {"relu": colors.COLORS["model"], "tanh": colors.COLORS["correct"],
           "sigmoid": colors.COLORS["error"]}
for act, hist in act_hist.items():
    ax.plot(hist, color=palette[act], lw=2, label=act)
ax.axhline(np.log(2), color=colors.COLORS["muted"], lw=1, ls=":", label="ln 2 = chance")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("Deep net (8 layers), SGD: sigmoid stalls where ReLU and tanh train")
ax.legend()
fig.tight_layout()
plt.show()

**Read the figure, honestly.** ReLU and tanh drive the loss down to near zero; **sigmoid stays flat at
`ln 2`** — the eight-layer net never leaves chance, because its gradient vanished on the way back (notebook
3), now surfaced as a *knob* you can flip. One honest caveat: a robust optimizer (**Adam**) partly rescues
the sigmoid net (its loss lands well below chance but far above ReLU — roughly 0.3–0.4), which is why you
rarely see this exact stall in practice. But the
real fix is **structural** — use ReLU, with good initialization and normalization — not a cleverer
optimizer. "Depth, not the optimizer" (notebook 3), restated as a setting.

## Initialization — `torch.nn.init`

Notebook 4 showed the initial weight *scale* must match the network's depth and activation, or the signal
dies or explodes before training starts. In torch, `torch.nn.init` sets it explicitly. On the same deep
ReLU net, we compare a too-**small** init, torch's **default**, and **He**.

In [ ]:
init_hist = {}
for init in ("small", "default", "he"):
    reset_seed()
    net = build_net(16, 8, activation="relu", init=init)
    tr_loss, _, tr, va = train(net, Xm_tr, ym_tr, Xm_va, ym_va, optimizer="sgd", lr=0.1, epochs=300)
    init_hist[init] = tr_loss
    print(f"  {init:8}: loss {tr_loss[0]:.3f} -> {tr_loss[-1]:.3f}   train {tr:.3f}  val {va:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
palette = {"small": colors.COLORS["error"], "default": colors.COLORS["highlight"],
           "he": colors.COLORS["model"]}
for init, hist in init_hist.items():
    ax.plot(hist, color=palette[init], lw=2, label=f"init = {init}")
ax.axhline(np.log(2), color=colors.COLORS["muted"], lw=1, ls=":", label="ln 2 = chance")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("Deep ReLU net, SGD: only He initialization gets it training")
ax.legend()
fig.tight_layout()
plt.show()

**Read the figure.** With a too-small init the loss is flat at `ln 2` — the signal died (notebook 4).
Strikingly, **torch's default init also stalls** this eight-layer net under plain SGD — the default is a
reasonable general choice, not a guarantee at depth. **He** initialization (`kaiming_normal_`) gets the
same net training within a few epochs. At depth, initialization is not a formality; it is one line,
`torch.nn.init.kaiming_normal_`, that decides whether the net trains at all.

## The optimizer menu

Notebook 7's loop used plain **SGD** — the `θ ← θ − η·∇L` step you built. Two upgrades converge faster and
more steadily — and their advantage grows with depth (the deep stalls of Figs 2–3 are where a plain
optimizer struggles most):

- **momentum** — accumulate a running average of past gradients (here, a weight of `0.9` on the running
  average), so the step rolls through small bumps and flat spots instead of stalling;
- **Adam** — keep a *per-parameter* adaptive step size (loosely, each weight gets its own learning rate).

They are one argument each in `torch.optim`. (momentum: Polyak 1964; Adam: Kingma & Ba 2015.)

In [ ]:
print(f"{'optimizer':10} {'final train loss':>18} {'val acc':>9} {'epochs to val-loss<0.35':>26}")
for opt_name in ("sgd", "momentum", "adam"):
    reset_seed()
    net = build_net(32, 2, activation="relu")
    tr_loss, va_loss, tr, va = train(net, Xm_tr, ym_tr, Xm_va, ym_va,
                                     optimizer=opt_name, lr=0.05, epochs=150)
    hit = next((i for i, v in enumerate(va_loss) if v < 0.35), None)
    print(f"{opt_name:10} {tr_loss[-1]:18.3f} {va:9.3f} {str(hit):>26}")

## Learning rate — the single most important knob

The learning rate `η` sets the step size. Too small and training crawls; too large and it overshoots and
oscillates. (Batch size is the other training knob: in a full training loop you iterate over the
mini-batches of chapter 11; here we pass the whole training set each step, to keep the picture clean.)

In [ ]:
print(f"{'learning rate':>14} {'final train loss':>18} {'val acc':>9}")
for lr in (1e-4, 1e-2, 1e-1, 1.0):
    reset_seed()
    net = build_net(32, 2, activation="relu")
    tr_loss, _, tr, va = train(net, Xm_tr, ym_tr, Xm_va, ym_va, optimizer="adam", lr=lr, epochs=150)
    print(f"{lr:14} {tr_loss[-1]:18.3f} {va:9.3f}")

## Regularizers as one-line layers

The dropout and batch-norm layers you built by hand (notebooks 5 and 6) are one line each in torch —
`nn.Dropout(p)` and `nn.BatchNorm1d` — dropped into the `nn.Sequential`. **This is where `.train()` and
`.eval()` start to matter** (notebook 7): dropout is active only in training, and batch norm uses the
batch's statistics while training but its running statistics at evaluation. Our `train` harness calls
`net.train()` and `net.eval()` at the right moments. We test them on the high-dimensional problem, where
a wide net on little data overfits hard.

In [ ]:
reg = {}
print(f"{'regularizer':20} {'train':>7} {'val':>7} {'gap':>7}")
for name, kw, wd in [("none", {}, 0.0), ("dropout 0.5", {"dropout": 0.5}, 0.0),
                     ("batch norm", {"batchnorm": True}, 0.0), ("weight decay 1e-2", {}, 1e-2)]:
    reset_seed()
    net = build_net(256, 2, activation="relu", in_dim=50, **kw)
    _, _, tr, va = train(net, Xh_tr, yh_tr, Xh_va, yh_va, optimizer="adam", lr=0.01, epochs=400,
                         weight_decay=wd)
    reg[name] = (tr, va)
    print(f"  {name:20} {tr:7.3f} {va:7.3f} {tr - va:+7.3f}")

In [ ]:
names = list(reg)
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9.5, 5))
ax.bar(x - 0.2, [reg[n][0] for n in names], 0.4, label="train", color=colors.COLORS["train"])
ax.bar(x + 0.2, [reg[n][1] for n in names], 0.4, label="validation", color=colors.COLORS["test"])
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=9)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel("accuracy")
ax.set_title("Regularizers keep train at 1.0 but lift validation (shrink the gap)")
ax.legend()
fig.tight_layout()
plt.show()

**Read the figure, honestly.** With no regularizer the net memorizes — train 1.0, validation far below:
the biggest gap. Every regularizer narrows it by trading a little training fit for generalization;
**dropout helps most here, batch norm least** (its win in notebook 6 was robustness and speed, not raw
held-out accuracy on a small net). *Which* regularizer wins is data- and capacity-dependent; the robust
fact is the direction — they all shrink the gap.

## Reading the loss curve: three failure modes

The train-vs-validation curve is your diagnostic. Three shapes, three different problems, three
different fixes:

- **Underfitting** — training accuracy never gets high (Fig 1, the tiny net). *Fix:* more capacity.
- **Overfitting** — training high, validation lagging behind (Fig 4, no regularizer). *Fix:* a
  regularizer, or less capacity.
- **Optimization failure** — the loss is **stuck at chance for essentially all of training** (Fig 3 small
  init, flat from the first epoch; Fig 2 sigmoid, after a brief initial dip onto `ln 2`).
  *Fix:* the activation, the initialization, or normalization — not more epochs.

The trap worth naming: a flat loss is **not** convergence. It is a network that never started (notebook 3).

## Honest tuning → one sealed test

To choose knobs, search them on a **validation** set — and touch the **test** set exactly once, at the
very end, to report the honest number. Tuning on the test set turns it into a training set and inflates
your estimate. We run a small grid over width and dropout on validation, pick the best, and read the
sealed test once.

In [ ]:
grid_best, best_cfg = -1.0, None
print(f"{'width':>6} {'dropout':>8} {'val acc':>9}")
for width in (64, 256):
    for dropout in (0.0, 0.5):
        reset_seed()
        net = build_net(width, 2, activation="relu", dropout=dropout, in_dim=50)
        _, _, _, va = train(net, Xh_tr, yh_tr, Xh_va, yh_va, optimizer="adam", lr=0.01, epochs=300)
        print(f"{width:6} {dropout:8} {va:9.3f}")
        if va > grid_best:
            grid_best, best_cfg = va, (width, dropout)

# Refit the chosen config and evaluate ONCE on the sealed test set.
reset_seed()
w, dp = best_cfg
net = build_net(w, 2, activation="relu", dropout=dp, in_dim=50)
train(net, Xh_tr, yh_tr, Xh_va, yh_va, optimizer="adam", lr=0.01, epochs=300)
with torch.no_grad():
    test_acc = (net(torch.tensor(Xh_te, dtype=torch.float32)).argmax(1)
                == torch.tensor(yh_te)).float().mean().item()
print(f"\nchosen on validation: width {w}, dropout {dp} (val {grid_best:.3f})")
print(f"SEALED TEST accuracy (read once): {test_acc:.3f}")

**Read the result, honestly.** The grid picks its best config on the validation set, and the **sealed
test** accuracy comes out close to the validation accuracy — so the choice generalizes, and picking on
validation did not inflate the estimate. Two honest notes: the winning configuration is **within split
noise** — reshuffle the data and a different cell of the grid may win (the chapter-11 capstone lesson) —
and the transferable part is the **discipline**, not the winner: search on validation, confirm once on a
sealed test.

## What you built

You turned every knob of the real PyTorch estimator, each from the concept that owns it:

- **capacity** (depth × width) — underfit → fit → overfit;
- **`activation`** — a tunable depth-pathology (deep sigmoid stalls at `ln 2`);
- **`torch.nn.init`** — He gets a deep net training where the default stalls;
- the **optimizer menu** — SGD → momentum → Adam;
- the **learning rate** — the single most important knob;
- **`nn.Dropout`** and **`nn.BatchNorm`** — the by-hand layers, now one line each.

And you learned to **read the loss curve** — underfitting, overfitting, optimization failure — and to
**tune honestly**: search on validation, report once on a sealed test. You can now drive a PyTorch model.

## Where this goes next, and your turn

**Notebook 9 — the capstone**: everything at once on **Fashion-MNIST**, end to end, with an honest verdict.

**Your turn** — each task *modifies the shown harness*; you never rewrite the loop.

1. **(warm-up)** Re-run the capacity sweep with `activation="tanh"`. Does the underfit → overfit picture
   change?
2. **(core)** Take the deep net that stalled at `init="small"` (Fig 3) and add batch norm
   (`batchnorm=True`); retrain with SGD. Does normalization rescue it where the small init failed
   (notebook 6)?
3. **(reach)** Add a learning-rate axis `{1e-3, 1e-2}` to the tuning grid and re-read the sealed test.
   Did the extra search change the winner — or only the validation number?

## References

- Kingma, D. P., & Ba, J. (2015). Adam: A Method for Stochastic Optimization. *ICLR* (arXiv:1412.6980).
- Polyak, B. T. (1964). Some methods of speeding up the convergence of iteration methods (momentum).
  *USSR Computational Mathematics and Mathematical Physics* 4(5).
- He, K., et al. (2015). Delving Deep into Rectifiers (He init; arXiv:1502.01852); Glorot & Bengio (2010),
  Understanding the difficulty of training deep networks (Xavier). Srivastava, N., et al. (2014), Dropout
  (*JMLR* 15). Ioffe, S., & Szegedy, C. (2015), Batch Normalization (arXiv:1502.03167). — recap from NB 4/5/6.
- Paszke, A., et al. (2019). PyTorch. *NeurIPS*.

*Previous:* **12.7 — hello-world in PyTorch.**  *Next:* **12.9 — capstone: Fashion-MNIST, end to end.**